In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from scipy.optimize import minimize
from termcolor import colored

## Import des données

In [16]:
raw_df = pd.read_parquet("data/reactions.parquet")
raw_df.head()

,id,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,conversation_a,conversation_b,model_pos,...,creative,complete,clear_formatting,incorrect,superficial,instructions_not_followed,model_pair_name,msg_rank,question_id,system_prompt
0,202099,2025-04-23 19:32:46.687338,gemma-3-12b,command-a,command-a,1,Quelle est la stratégie commerciale du champag...,[{'content': 'Quelle est la stratégie commerci...,[{'content': 'Quelle est la stratégie commerci...,b,...,False,True,False,False,False,False,"[command-a, gemma-3-12b]",0,1a62544c50474085b25240991e9fc620-6e6ffca415f44...,NaN
1,176467,2025-03-25 13:41:21.988182,llama-3.1-8b,claude-3-5-sonnet-v2,llama-3.1-8b,1,Qui est président en France ?,"[{'content': 'Qui est président en France ?', ...","[{'content': 'Qui est président en France ?', ...",a,...,False,True,False,False,False,False,"[claude-3-5-sonnet-v2, llama-3.1-8b]",0,ba089364a6e64d14b80eae6f7dd1885f-cda152aa7fe04...,NaN
2,231166,2025-06-03 15:38:11.580715,gpt-4.1-nano,llama-4-scout,llama-4-scout,1,tests logiciel,"[{'content': 'tests logiciel', 'metadata': Non...","[{'content': 'tests logiciel', 'metadata': Non...",b,...,False,True,False,False,False,False,"[gpt-4.1-nano, llama-4-scout]",0,e46e59300e8545038bfc22dfffb0b746-b006443eb1564...,NaN
3,264589,2025-08-20 13:52:02.676745,llama-3.1-405b,mistral-large-2411,llama-3.1-405b,13,Vols au meilleur prix au départ de Marseille p...,[{'content': 'Vols au meilleur prix au départ ...,[{'content': 'Vols au meilleur prix au départ ...,a,...,False,False,False,False,False,False,"[llama-3.1-405b, mistral-large-2411]",6,1f181e340b664f88b1aacc6770d0eeff-3f295419957c4...,NaN
4,203152,2025-04-25 20:40:31.489534,o4-mini,claude-3-7-sonnet,o4-mini,1,Comporte toi comme un expert en rédaction de p...,[{'content': 'Comporte toi comme un expert en ...,[{'content': 'Comporte toi comme un expert en ...,a,...,False,False,False,False,True,False,"[claude-3-7-sonnet, o4-mini]",0,5a646fd23c7c480dbbb9dac5f50b8c6d-5d343345ef294...,NaN


In [17]:
col = list(raw_df.columns)
to_keep = ["model_a_name", "model_b_name", "model_pos", "liked", "creative", "useful", "incorrect"]


df = raw_df[to_keep]
del raw_df
df.head()

,model_a_name,model_b_name,model_pos,liked,creative,useful,incorrect
0,gemma-3-12b,command-a,b,True,False,False,False
1,llama-3.1-8b,claude-3-5-sonnet-v2,a,True,False,False,False
2,gpt-4.1-nano,llama-4-scout,b,True,False,False,False
3,llama-3.1-405b,mistral-large-2411,a,True,False,True,False
4,o4-mini,claude-3-7-sonnet,a,False,False,False,False


## Biais de position A/B

In [38]:
from scipy.stats import chi2_contingency

ALPHA = 0.05

def cramers_v(crosstab):
    stat, _, dof, _ = chi2_contingency(crosstab)
    n = crosstab.values.sum()
    return np.sqrt(stat / (n * dof))

def chi_2_test(variable:str):
    # Test du chi2 pour la variable "liked"
    print(colored(f"Test du chi2 pour la variable '{variable}'", "blue"))
    contingency_table = pd.crosstab(df["model_pos"], df[variable])
    print(contingency_table, end=f"\n{'-'*25}\n")
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    cramers_v_value = cramers_v(contingency_table)
    print(f"Chi2: {chi2:.3f}, p-value: {p:.2E}, dof: {dof}, Cramer's V: {cramers_v_value:.2E}")

    # Interprétation selon cramers_v_value
    if cramers_v_value < 0.1:
        print(colored(f"Association très faible ou nulle ({cramers_v_value:.2E} < 0.1)", "green"))
    elif cramers_v_value < 0.3:
        print(colored(f"Association faible ({cramers_v_value:.2E} < 0.3)", "yellow"))
    elif cramers_v_value < 0.5:
        print(colored(f"Association modérée ({cramers_v_value:.2E} < 0.5)", "yellow"))
    else:
        print(colored(f"Association forte ({cramers_v_value:.2E} >= 0.5)", "red"))

chi_2_test("liked")
print()
chi_2_test("creative")
print()
chi_2_test("useful")
print()
chi_2_test("incorrect")

Test du chi2 pour la variable 'liked'
liked      False  True 
model_pos              
a          14890  30602
b          14746  31595
-------------------------
Chi2: 8.663, p-value: 3.25E-03, dof: 1, Cramer's V: 9.71E-03
Association très faible ou nulle (9.71E-03 < 0.1)

Test du chi2 pour la variable 'creative'
creative   False  True 
model_pos              
a          42270   2965
b          42881   3201
-------------------------
Chi2: 5.499, p-value: 1.90E-02, dof: 1, Cramer's V: 7.76E-03
Association très faible ou nulle (7.76E-03 < 0.1)

Test du chi2 pour la variable 'useful'
useful     False  True 
model_pos              
a          34765  10470
b          34803  11279
-------------------------
Chi2: 22.186, p-value: 2.48E-06, dof: 1, Cramer's V: 1.56E-02
Association très faible ou nulle (1.56E-02 < 0.1)

Test du chi2 pour la variable 'incorrect'
incorrect  False  True 
model_pos              
a          40200   5035
b          41049   5033
-------------------------
Chi2: 0.995, p-

En analysant pour chaque variable (liked, creative, useful, incorrect) la proportion de votes pour le modèle en position A, on n'observe pas de biais de position significatif sur la base du V de Cramer.

On peut donc considérer qu'il n'y a pas de biais significatif dans les votes en fonction de la position du modèle (A ou B) pour ces variables/

## 3.3. Protocole d'annotations amélioré

Pour garantir un taux d'abandon $<15\%$, chaque proposition d'amélioration du protocole sera associé à un code qui juge qualitiativement de son impact supposé sur le risque d'abandon (*RA1*: faible, *RA2*: modéré, *RA3*: élevé, *NC*: non concerné).

### 3.3.1. Biais de position: Rotation systématique et masquage renforcé

Le biais de position (préférence pour la réponse A) est bien documenté dans la littérature sur les arenas. Zheng et al. (2023) dans Judging LLM-as-a-Judge montrent que les juges humains comme LLM favorisent systématiquement la première réponse affichée, avec un effet allant jusqu'à 10-15 points de taux de sélection.

**Mesures proposées :**
- **Rotation aléatoire à chaque session** (*RA1*) : l'assignation A/B doit être tirée aléatoirement et indépendamment pour chaque paire.
- **Affichage simultané vertical** plutôt que côte à côte (*RA1*) : Karpinska et al. (2021) montrent dans le cadre des évaluations de traduction que la disposition verticale réduit les biais de primauté visuels.
- **Indicateurs neutres** (*RA1*) : remplacer "Réponse A / Réponse B" par des labels non ordonnés et rotatifs (ex. couleurs ou formes).

### 3.3.2. Biais de longueur: Normalisation perceptuelle

Dubois et al. (2024) dans Length-Controlled AlpacaEval montrent que les évaluateurs humains et automatiques favorisent les réponses plus longues indépendamment de leur qualité, avec une corrélation longueur/préférence pouvant atteindre r = 0.7.

**Mesures proposées :**

- **Troncature affichée égalisée** (*RA2*) : afficher les deux réponses avec le même nombre de caractères visibles initialement, avec un bouton "voir plus" identique pour les deux — ce qui force l'évaluation du contenu initial et non du volume.
- **Bandeau d'avertissement** (*RA2*) : afficher un message "les réponses peuvent différer en longueur — évaluez la qualité, pas la quantité" (nudge cognitif documenté par Kiritchenko & Mohammad, 2017).
- **Annotation de la densité** (*RA3*): ajouter une dimension "La réponse était-elle trop longue / trop courte / appropriée ?" pour capturer explicitement ce signal et le débruiter en aval.

### 3.3.2. Biais de sélection: Diversification des tâches et des utilisateurs

Chiang et al. (2024) dans l'analyse de Chatbot Arena soulignent que les utilisateurs volontaires sur-représentent certains profils (développeurs, anglophones, tâches de coding), ce qui biaise les rankings vers les modèles performants sur ces niches. Notamment pour Compar:IA, les utilisateur·ices qui posent des questions "créatives" sont sur-représenté·es dans le dataset comparia-reactions.

**Mesures proposées :**

- **Recrutement de panels ciblés** (*NC*) : compléter les contributions spontanées par des panels recrutés sur des profils sous-représentés, à l'image du protocole de Scale AI / LMSYS pour les évaluations multilingues (ce n'est pas forcément dans l'ADN du projet Compar:IA, et cela peut représenter un coût important)
- **Pondération post-hoc** (*NC*) : appliquer une pondération inverse à la fréquence de chaque profil utilisateur dans le calcul du score final (approche IPW, Swaminathan & Joachims, 2015).

### Sources

- Zheng et al. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.
- Dubois et al. (2024). Length-Controlled AlpacaEval.
- Chiang et al. (2024). Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference.
- Karpinska et al. (2021). The Perils of Using Mechanical Turk to Evaluate Open-Ended Text Generation.
- Swaminathan & Joachims (2015). Counterfactual Risk Minimization.